# Automata Extraction from RNN (Tomita 2)
This notebook demonstrates how to extract a DFA from a trained RNN using the L* algorithm via the `aalpy` library.

## 1. DATA PREPARATION & RNN TRAINING

This cell trains a PyTorch RNN on Tomita Grammar 2. 
Optimizations included:
- **Batch Processing**: Sequences are padded and processed in parallel using a `DataLoader`.
- **Stability**: Gradient clipping and the Adam optimizer ensure smooth convergence.
- **Efficiency**: Dataset size and lengths are tuned for fast training.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
import re, random

# Tomita Grammar 2: (0*11)*0*
def is_tomita2(s): return re.fullmatch(r'(0*11)*0*', s) is not None

def generate_balanced_data(num_samples=1500, max_len=15):
    data, seen, pos, neg = [], set(), 0, 0
    target = num_samples // 2
    while len(data) < num_samples:
        s = ''.join(random.choice(['0', '1']) for _ in range(random.randint(0, max_len)))
        if s in seen: continue
        label = 1.0 if is_tomita2(s) else 0.0
        if (label == 1.0 and pos < target) or (label == 0.0 and neg < target):
            data.append((s, label))
            if label == 1.0: pos += 1
            else: neg += 1
            seen.add(s)
    return data

class TomitaDataset(Dataset):
    def __init__(self, data):
        self.samples = [(torch.eye(2)[[int(c) for c in s]] if s else torch.zeros(0, 2), torch.tensor([l])) for s, l in data]
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def collate_fn(batch):
    xs, ys = zip(*batch)
    lengths = torch.clamp(torch.tensor([len(x) for x in xs]), min=1)
    # Pad sequences with zeros; handle empty strings by padding to length 1
    xs_padded = pad_sequence([x if len(x)>0 else torch.zeros(1, 2) for x in xs], batch_first=True)
    return xs_padded, torch.stack(ys), lengths

class SimpleRNN(nn.Module):
    def __init__(self, hidden_size=10):
        super().__init__()
        self.rnn = nn.RNN(2, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x, lengths=None):
        if lengths is not None: x = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, h = self.rnn(x)
        return self.sigmoid(self.fc(h.squeeze(0)))

# For Oracle compatibility (Cell 2/3)
def to_tensor(s):
    t = torch.zeros(1, len(s) if s else 1, 2)
    for i, char in enumerate(s): t[0, i, int(char)] = 1.0
    return t

# Training
train_data = generate_balanced_data()
loader = DataLoader(TomitaDataset(train_data), batch_size=64, shuffle=True, collate_fn=collate_fn)
model = SimpleRNN()
optimizer, criterion = optim.Adam(model.parameters(), lr=0.001), nn.BCELoss()

for epoch in range(200):
    total_loss, correct = 0, 0
    for xb, yb, lens in loader:
        optimizer.zero_grad()
        preds = model(xb, lens)
        loss = criterion(preds, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += ((preds > 0.5) == yb).sum().item()
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}, Acc: {correct/len(train_data)*100:.2f}%')

Epoch 20, Loss: 0.3620, Acc: 91.20%
Epoch 40, Loss: 0.1938, Acc: 93.67%
Epoch 60, Loss: 0.1324, Acc: 95.80%
Epoch 80, Loss: 0.0918, Acc: 96.93%
Epoch 100, Loss: 0.0629, Acc: 97.73%
Epoch 120, Loss: 0.0432, Acc: 99.07%
Epoch 140, Loss: 0.0294, Acc: 99.80%
Epoch 160, Loss: 0.0192, Acc: 99.93%
Epoch 180, Loss: 0.0137, Acc: 99.93%
Epoch 200, Loss: 0.0095, Acc: 99.93%


## 2. ORACLE WRAPPER (SUL)

In [2]:
from aalpy.base import SUL

class RNNSUL(SUL):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.current_seq = ""
        
    def pre(self):
        self.current_seq = ""
    
    def post(self):
        pass
        
    def step(self, letter):
        if letter is None: return False
        self.current_seq += str(letter)
        with torch.no_grad():
            pred = self.model(to_tensor(self.current_seq))
            return pred.item() > 0.5

sul = RNNSUL(model)

## 3. L* EXTRACTION & IMAGE-BASED VISUALIZATION

In [3]:
from aalpy.learning_algs import run_Lstar
from aalpy.oracles import RandomWMethodEqOracle
from IPython.display import display, Image
import requests
from typing import Any

import urllib.parse

def render_dfa_image(dfa):
    """Generates a clickable link for the DFA and prints a text fallback."""
    dot = "digraph DFA {\n  rankdir=LR;\n  node [shape=circle, fontname=Helvetica];\n"
    
    for state in dfa.states:
        shape = "doublecircle" if state.output else "circle"
        dot += f'  "{state.state_id}" [shape={shape}];\n'
        for char, next_st in state.transitions.items():
            dot += f'  "{state.state_id}" -> "{next_st.state_id}" [label="{char}"];\n'
    dot += "}"
    
    print("------------------------------------\n")
    
    # 2. Varianta Link Clickabil
    encoded_dot = urllib.parse.quote(dot)
    url = f"https://quickchart.io/graphviz?graph={encoded_dot}"
    
    print(url)

alphabet = ['0', '1']
eq_oracle = RandomWMethodEqOracle(alphabet, sul, walks_per_state=20, walk_len=10)

# Rulam L* (va printa intrebarile MQ si EQ in consola)
res: Any = run_Lstar(alphabet, sul, eq_oracle, automaton_type='dfa', print_level=2)
extracted_dfa = res[0] if isinstance(res, tuple) else res

if extracted_dfa:
    print(f"\nExtracted Automaton with {len(extracted_dfa.states)} states.")
    if len(extracted_dfa.states) in [3, 4, 5]: # Tomita 2 are de obicei 4 stari (Start, Odd, Even, Dead)
        print("SUCCESS: Minimal DFA found!")
    render_dfa_image(extracted_dfa)
else:
    print("Extraction failed.")

Hypothesis 1: 2 states.
Hypothesis 2: 3 states.
Hypothesis 3: 4 states.
-----------------------------------
Learning Finished.
Learning Rounds:  3
Number of states: 4
Time (in seconds)
  Total                : 0.13
  Learning algorithm   : 0.01
  Conformance checking : 0.12
Learning Algorithm
 # Membership Queries  : 17
 # MQ Saved by Caching : 16
 # Steps               : 45
Equivalence Query
 # Membership Queries  : 80
 # Steps               : 567
-----------------------------------

Extracted Automaton with 4 states.
SUCCESS: Minimal DFA found!
------------------------------------

https://quickchart.io/graphviz?graph=digraph%20DFA%20%7B%0A%20%20rankdir%3DLR%3B%0A%20%20node%20%5Bshape%3Dcircle%2C%20fontname%3DHelvetica%5D%3B%0A%20%20%22s0%22%20%5Bshape%3Dcircle%5D%3B%0A%20%20%22s0%22%20-%3E%20%22s1%22%20%5Blabel%3D%220%22%5D%3B%0A%20%20%22s0%22%20-%3E%20%22s2%22%20%5Blabel%3D%221%22%5D%3B%0A%20%20%22s1%22%20%5Bshape%3Ddoublecircle%5D%3B%0A%20%20%22s1%22%20-%3E%20%22s1%22%20%5Blabel%3

## 4. SAVE MODEL STATE
We save the model state dictionary for later use in a demo.

In [4]:
save_path = '../models/tomita2_rnn.pth'
checkpoint = {
    'model_state_dict': model.state_dict(),
    'hidden_size': 10 # Default used in class definition
}
torch.save(checkpoint, save_path)
print(f"Model saved to {save_path}")

Model saved to ../models/tomita2_rnn.pth
